# scRNA-seq

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
from pathlib import Path
import scvi
import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sn
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import itertools

scvi.settings.seed = 30

### Annotate Single Cells with Cell Type Label

In [ ]:
######################################################## load reference
ref_data = sc.read_h5ad('TS_Mammary.h5ad')
print(ref_data)

######################################################## load data
file = 'scrnaseq_processed_data_10232025.h5ad' ### <--- data

dat1 = sc.read_h5ad(file)
#dat1.obs.to_csv("dat1.csv")
print(dat1)
print("After reload:", dat1.obs['Sample'].unique())

######################################################## concatenate data
dat1 = dat1.concatenate(ref_data)

In [ ]:
dat1

In [ ]:
dat1.obs

In [ ]:
dat1.X

In [ ]:
dat1_copy = dat1.copy()

In [ ]:
#dat1_copy.obs.to_csv("dat1_copy_obs.csv")

In [ ]:
######################################################## normalize data
dat1.layers["counts"] = dat1.X.copy()
sc.pp.normalize_total(dat1, target_sum = 10000)
sc.pp.log1p(dat1)
dat1.raw = dat1

print(dat1)
######################################################## filter
sc.pp.highly_variable_genes(dat1, flavor = 'seurat_v3', n_top_genes=4000, layer = 'counts', subset = True)
print(dat1)

In [ ]:
######################################################## train model
scvi.model.SCVI.setup_anndata(dat1, layer = 'counts')
vae = scvi.model.SCVI(dat1) ### initialize the model
vae.train()

######################################################## predict label
dat1.obs['cell_ontology_class'] = dat1.obs['cell_ontology_class'].cat.add_categories('Unknown')
dat1.obs = dat1.obs.fillna(value = {'cell_ontology_class': 'Unknown'})

lvae = scvi.model.SCANVI.from_scvi_model(vae, adata = dat1, unlabeled_category = 'Unknown', labels_key = 'cell_ontology_class')
lvae.train()

dat1.obs['predicted'] = lvae.predict(dat1)
dat1.obs['bc2'] = dat1.obs.index.map(lambda x: x[:-2])

######################################################## store labels in a dictionary
cell_mapper = dict(zip(dat1.obs.bc2, dat1.obs.predicted))

######################################################## map to data
dat1 = sc.read_h5ad(file)
dat1.var_names_make_unique()
dat1.obs['cell_type'] = dat1.obs.index.map(cell_mapper)
dat1.obs.groupby('Sample').count()
dat1.obs

### Data Analysis

In [ ]:
dat1

In [ ]:
############################### remove cells that do not meet the required minimum number of genes
sc.pp.filter_cells(dat1, min_genes = 200) ### <-- 100

############################### remove genes that do not meet the required minimum number of cells
sc.pp.filter_genes(dat1, min_cells = 10) ### <--- 10
dat1

In [ ]:
######################################################## normalize data
dat1.layers['counts'] = dat1.X.copy()
sc.pp.normalize_total(dat1, target_sum = 10000)
sc.pp.log1p(dat1)
dat1.raw = dat1

print(dat1)
######################################################## filter
sc.pp.highly_variable_genes(dat1, flavor='seurat_v3', n_top_genes=2000, layer='counts', subset=True)
print(dat1)

In [ ]:
######################################################## train model
scvi.model.SCVI.setup_anndata(dat1, layer='counts',
                              categorical_covariate_keys=['Sample'],
                              #continuous_covariate_keys=['pct_counts_mt', 'total_counts', 'pct_counts_ribo'])
                              continuous_covariate_keys=['total_counts', 'pct_counts_ribo'])

model = scvi.model.SCVI(dat1)
model.train()

######################################################## prepare data for clustering
model.get_latent_representation().shape
dat1.obsm['X_scVI'] = model.get_latent_representation()
dat1.layers['scvi_normalized'] = model.get_normalized_expression(library_size=10000)

sc.pp.neighbors(dat1, use_rep = 'X_scVI')

In [ ]:
#dat10_copy = dat1.copy()

In [ ]:
#dat1 = dat10_copy

In [ ]:
import torch, random, numpy as np
seed = 30
scvi.settings.seed = seed
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [ ]:
#dat10_copy.obs
#dat10_copy.obs.to_csv("dat10_obs.csv")
#dat1.write_h5ad('scrnaseq_processed_data_10232025_FINAL.h5ad')

In [ ]:
######################################################## plot umap, based on leiden clustering
sc.tl.umap(dat1)
sc.tl.leiden(dat1, resolution = 0.6)

sc.pl.umap(dat1, color = ['leiden'], frameon = False, title = "", save='UMAP_cell_type_scrnaseq.png') ### leiden clustering
sc.pl.umap(dat1, color = ['Sample'], frameon = False, title = "", save='UMAP_sample_scrnaseq.png') ### leiden clustering

In [ ]:
######################################################## R Shiny
umap_df = pd.DataFrame({
    "cell_id": dat1.obs_names,
    "UMAP1": dat1.obsm['X_umap'][:, 0], 
    "UMAP2": dat1.obsm['X_umap'][:, 1],
    "leiden": dat1.obs['leiden'].values
})
umap_df.to_csv("data_umap_leiden_1.csv", index=False)

In [ ]:
######################################################## plot umap, based on labelled cell type annotation from Tabula Sapiens Consortum
sc.pl.umap(dat1, color = 'cell_type', frameon = False, title = "")

In [ ]:
dat1

In [ ]:
######################################################## plot biomarker genes
fig, axs = plt.subplots(4,2, figsize=(12,16))

sc.pl.umap(dat1, color = ['CD68'], frameon = False, ax=axs[0,0], show=False) ### macrophage
sc.pl.umap(dat1, color = ['CD3E'], frameon = False, ax=axs[0,1], show=False) ### T cells
sc.pl.umap(dat1, color = ['FOXP3'], frameon = False, ax=axs[1,0], show=False) ### CD4+ regulatory T cell (Tregs)
sc.pl.umap(dat1, color = ['CXCL13'] , frameon = False, ax=axs[1,1], show=False) ### CD4+ follicular helper cell (Tfh)
sc.pl.umap(dat1, color = ['CD8A'], frameon = False, ax=axs[2,0], show=False) ### CD8+ T cell
sc.pl.umap(dat1, color = ['NCAM1'], frameon = False, ax=axs[2,1], show=False) ### NK cell
sc.pl.umap(dat1, color = ['MS4A1'], frameon = False, ax=axs[3,0], show=False) ### B cell
sc.pl.umap(dat1, color = ['TNFRSF17'], frameon = False, ax=axs[3,1], show=False) ### plasma cell
#plt.delaxes(axs[3,1])
plt.savefig('biomarkers_scrnaseq_i.png')

In [ ]:
######################################################## plot biomarker genes
fig, axs = plt.subplots(3,2, figsize=(12,12))

sc.pl.umap(dat1, color = ['PECAM1'] , frameon = False, ax=axs[0,0], show=False) ### endothelial cell
sc.pl.umap(dat1, color = ['COL1A1'] , frameon = False, ax=axs[0,1], show=False) ### fibroblast cells COL1A1
sc.pl.umap(dat1, color = ['PDGFRA'], frameon = False, ax=axs[1,0], show=False) ### ECM fibroblast, PDGFRA, DCN
sc.pl.umap(dat1, color = ['POSTN'], frameon = False, ax=axs[1,1], show=False) ### myCAF, POSTN, TAGLN
sc.pl.umap(dat1, color = ['RGS5'], frameon = False, ax=axs[2,0], show=False) ### vascular CAF (vCAF)
plt.delaxes(axs[2,1])
plt.savefig('biomarkers_scrnaseq_ii.png')

In [ ]:
######################################################## plot biomarker genes
fig, axs = plt.subplots(3,2, figsize=(12,12))

sc.pl.umap(dat1, color = ['EPCAM'] , frameon = False, ax=axs[0,0], show=False) ### epithelial cells
sc.pl.umap(dat1, color = ['SCGB2A2'] , frameon = False, ax=axs[0,1], show=False) ### luminal L1 cell
sc.pl.umap(dat1, color = ['KRT23'], frameon = False, ax=axs[1,0], show=False) ### luminal L2 cell
sc.pl.umap(dat1, color = ['MLLT4'], frameon = False, ax=axs[1,1], show=False) ### junctional epithelial cell
sc.pl.umap(dat1, color = ['DKK1'], frameon = False, ax=axs[2,0], show=False) ### tumor epithelial cell
plt.delaxes(axs[2,1])
plt.savefig('biomarkers_scrnaseq_iii.png')

In [ ]:
######################################################## compute differential expression by Wilcoxon test
sc.tl.rank_genes_groups(dat1, 'leiden', method="wilcoxon")
sc.pl.rank_genes_groups(dat1, n_genes=25, sharey=False)

######################################################## filter by adjusted p-values and log-fold-changes from Wilcoxon test
markers_all = sc.get.rank_genes_groups_df(dat1, None)
markers_all.to_csv('biomarker_raw.csv', index=False)
markers = markers_all[(markers_all.pvals_adj < 0.05) & (markers_all.logfoldchanges > 0.5)]
#markers.to_csv('biomarker_processed.csv', index=False)

In [ ]:
#df = sc.get.rank_genes_groups_df(dat1, group='0')
#print(df)
#df.to_csv("cluster0_ranked_genes.csv", index=False)

In [ ]:
######################################################## compute differential expression by scvi-DE
markers_scvi = model.differential_expression(adata=dat1, groupby='leiden')

######################################################## filter by FDR and log-fold-change mean from scvi-DE
markers_scvi = markers_scvi[(markers_scvi['is_de_fdr_0.05']) & (markers_scvi.lfc_mean > 0.5)]

In [ ]:
markers_scvi

In [ ]:
sc.pl.umap(dat1, color = ['leiden'], frameon = False, legend_loc = "on data")

In [ ]:
dat1

In [ ]:
######################################################## labelling leiden clusters
Type = {
"0":'vCAF',
"1":'Macrophage',
"2":'CD8+ T Cell',
"3":'Endothelial Cell',
"4":'MyCAF',
"5":'Junctional Epithelial Cell',
"6":'CD4+ Tregs Cell',
"7":'Luminal L1 Cell',
"8":'Plasma Cell',
"9":'Luminal L2 Cell',
"10":'CD4+ Tfh Cell',
"11":'ECM Fibroblast',
"12":'Tumor Epithelial Cell',
"13":'NK cell',
"14":'B cell'    
}
dat1.obs['Type'] = dat1.obs.leiden.map(Type)

######################################################## plot
sc.pl.umap(dat1, color = ['Type'], frameon = False, title="",  save='UMAP_scrnaseq.png')

In [ ]:
######################################################## wilcoxon DE
######################################################## Load APA gene list
polyadb_list = pd.read_csv('polyadb4_TR_gene_list.csv')['gene_symbol'].tolist()

######################################################## filter for adjusted p-value < 0.05 and log-fold-change > 0.5
de_genes_w = markers[(markers['pvals_adj'] < 0.05) & (markers['logfoldchanges'] > 0.5)].copy()
print(de_genes_w)
num_unique_genes = len(de_genes_w['names'].unique())
print(f"Number of significant DE genes (Wilcoxon): {num_unique_genes}")

######################################################## subset for HVGs
hvg_genes = dat1.var_names[dat1.var['highly_variable']]
de_genes_w_hvg = de_genes_w[de_genes_w['names'].isin(hvg_genes)].copy()
num_unique_hvg_de = len(de_genes_w_hvg['names'].unique())
print(f"Number of significant DE genes belonging to HVGs: {num_unique_hvg_de}")

######################################################## subset for APA genes
apa_de_genes_w = [g for g in polyadb_list if g in de_genes_w_hvg['names'].values]
num_unique_hvg_de_apa = len(set(apa_de_genes_w))
print(f"Number of APA DE genes belonging to HVGs: {num_unique_hvg_de_apa}")

######################################################## save
de_genes_w_hvg.to_csv("de_genes_w_hvg_filtered.csv", index=False)

matched_w = dat1[:, apa_de_genes_w]

In [ ]:
######################################################## scvi-DE
######################################################## Load APA gene list
polyadb_list = pd.read_csv('polyadb4_TR_gene_list.csv')['gene_symbol'].tolist()

######################################################## DE (markers_scvi)
markers_scvi.index.name = 'gene'

######################################################## filter for DE genes
de_genes = markers_scvi[(markers_scvi['is_de_fdr_0.05']) &(markers_scvi['lfc_mean'] > 0.5)].copy()

######################################################## subset for APA genes
apa_de_genes = [g for g in polyadb_list if g in de_genes.index and g in dat1.var_names]

######################################################## sort by lfc_mean
apa_de_genes = (de_genes.loc[apa_de_genes].sort_values('lfc_mean', ascending=False).index.tolist())
#pd.DataFrame(apa_de_genes, columns=['gene']).to_csv("apa_de_genes_sorted.csv", index=False)

######################################################## subet HVGs
hvg_genes = dat1.var_names[dat1.var['highly_variable']]
apa_de_genes_hvg = [g for g in apa_de_genes if g in hvg_genes]
#num_unique_apa_hvg = len(set(apa_de_genes_hvg))
#print(f"Number of unique APA DE genes within HVGs: {num_unique_apa_hvg}")

########################################################
matched_s = dat1[:, apa_de_genes_hvg]
print(f"Number of APA DE genes belonging to HVGs (SVCI): {len(matched_s.var_names.unique().tolist())}")
print("APA genes:", matched_s.var_names.unique().tolist())
#sc.pl.dotplot(matched,var_names=apa_de_genes, groupby="cell_type_2", swap_axes=True,save='_apa_genes_from_scvi_DE.png')

In [ ]:
######################################################## genes that are present in both lists
genes_matched_w = matched_w.var_names.unique().tolist()
genes_matched_s = matched_s.var_names.unique().tolist()
print("Wilcoxon:", len(genes_matched_w))
print("SCVI:", len(genes_matched_s))

########################################################
shared_genes = [gene for gene in genes_matched_w if gene in genes_matched_s]

print(f"Number of shared genes: {len(shared_genes)}")
print("Shared genes:", shared_genes)

########################################################
shared_matched = dat1[:, shared_genes]
shared_matched.var_names_make_unique()

######################################################## plot heatmap
sc.pl.heatmap(
    shared_matched,
    var_names=shared_genes,
    groupby="Type",
    use_raw=False,
    cmap='RdBu_r',          
    standard_scale='var',   ### based on z-score
    swap_axes=True,
    dendrogram=True,        ### based on hierarchical clustering
    save="heatmap_stats_scrnaseq.png"
)

In [ ]:
######################################################## plot violin plot
genename="TIMP3"
timp3_data = dat1[:, [genename]].copy()
########################################################
sc.pl.violin(timp3_data, keys=genename, groupby="Type", rotation=90, scale="width", inner="box", jitter=0.4, save=f"violin_{genename}.png")

In [ ]:
######################################################## perform pairwise Mann–Whitney U tests
genename="TIMP3"
sdata = dat1[:, [genename]].copy()

expr = sdata[:, genename].X
if not isinstance(expr, np.ndarray):
    expr = expr.toarray().flatten()
groups = sdata.obs[groupby].values

plot_df = pd.DataFrame({"expression": expr,"group": groups})

######################################################## compute data
unique_groups = np.unique(groups)
pair_results = []

for g1, g2 in itertools.combinations(unique_groups, 2):
    x = plot_df.loc[plot_df['group'] == g1, 'expression']
    y = plot_df.loc[plot_df['group'] == g2, 'expression']
    stat, pval = mannwhitneyu(x, y, alternative='two-sided')
    pair_results.append({"group1": g1, "group2": g2, "pval": pval})

######################################################## adjust p-values
pvals = [r['pval'] for r in pair_results]
adj_pvals = multipletests(pvals, method='fdr_bh')[1]

for i, r in enumerate(pair_results):
    r['pval_adj'] = adj_pvals[i]

pair_results_df = pd.DataFrame(pair_results)

########################################################
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', None)

pair_results_df = pair_results_df.sort_values('pval_adj').reset_index(drop=True)

print(pair_results_df)

pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.width')
pd.reset_option('display.max_colwidth')
#pair_results_df.to_csv("TIMP3_pairwise_stats.csv", index=False)